# Model Training

The objective of this notebook is to train and compare multiple machine learning models for forecasting demand 8 weeks ahead.

Since this is a time-series forecasting problem, a chronological train-test split is used to prevent data leakage and simulate real-world forecasting.

# Import all necessary libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

from sklearn.linear_model import LinearRegression, Ridge

from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import RandomizedSearchCV

from xgboost import XGBRegressor

# Load the Dataset

In [ ]:
df = pd.read_csv("../data/processed/final_dataset.csv")

In [ ]:
print(df.shape)
df.head()

In [ ]:
# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

In [ ]:
split_date = df["date"].quantile(0.80)

train_df = df[df["date"] < split_date]

test_df = df[df["date"] >= split_date]

In [ ]:
train_dates = train_df["date"].reset_index(drop=True)
test_dates = test_df["date"].reset_index(drop=True)

# Training and Test Dataset

In [ ]:
FEATURES = [
    col for col in df.columns
    if col not in ["date", "target"]
]

TARGET = "target"

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

In [ ]:
print("\nTraining Shape:", X_train.shape)
print("Testing Shape :", X_test.shape)

In [ ]:
print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))

print("\nTrain Date Range:")
print(train_df["date"].min(), "to", train_df["date"].max())

print("\nTest Date Range:")
print(test_df["date"].min(), "to", test_df["date"].max())

# Baseline Prediction

In [ ]:
baseline_predictions = X_test["demand"]

# Evaluate model function

In [ ]:
def evaluate_model(model_name, y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    r2 = r2_score(y_true, y_pred)
    
    mape = mean_absolute_percentage_error(y_true,y_pred)

    print(model_name)

    print(f"MAE  : {mae:.2f}")
    print(f"RMSE : {rmse:.2f}")
    print(f"R²   : {r2:.4f}")
    print(f"MAPE : {mape*100:.2f}%")
    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE": mape*100
    }

In [ ]:
baseline_results = evaluate_model(
    "Baseline Model",
    y_test,
    baseline_predictions
)

# Linear Regression Model

In [ ]:
lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

lr_predictions = lr_model.predict(X_test)

lr_results = evaluate_model(

    "Linear Regression",

    y_test,

    lr_predictions

)

# Ridge Model

In [ ]:
ridge_model = Ridge(alpha=1.0)

ridge_model.fit(X_train, y_train)

ridge_predictions = ridge_model.predict(X_test)

ridge_results = evaluate_model(

    "Ridge Regression",

    y_test,

    ridge_predictions

)

# Random Forest Model with Hyperparameter tuning

In [ ]:
rf_params = {

    "n_estimators": [100, 200, 300, 500],

    "max_depth": [5, 10, 15, None],

    "min_samples_split": [2, 5, 10],

    "min_samples_leaf": [1, 2, 4]

}

rf_random = RandomizedSearchCV(

    estimator=RandomForestRegressor(random_state=42),

    param_distributions=rf_params,

    n_iter=15,

    scoring="neg_root_mean_squared_error",

    cv=3,

    random_state=42,

    n_jobs=-1,

    verbose=1

)

rf_random.fit(X_train, y_train)

print("Best RF Parameters:")

print(rf_random.best_params_)

rf_model = rf_random.best_estimator_

In [ ]:
rf_predictions = rf_model.predict(X_test)

In [ ]:
rf_results = evaluate_model(

    "Random Forest (Tuned)",

    y_test,

    rf_predictions

)

# Important Features

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

plt.figure(figsize=(10,6))

plt.barh(
    feature_importance["Feature"][:10],
    feature_importance["Importance"][:10]
)

plt.gca().invert_yaxis()

plt.xlabel("Importance")
plt.title("Top 10 Feature Importances (Random Forest)")

plt.tight_layout()

plt.savefig("../outputs/figures/feature_importance.png")

plt.show()

# XGB Model 

In [ ]:
xgb_params = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.2]
}

xgb_random = RandomizedSearchCV(
    estimator=XGBRegressor(
        objective="reg:squarederror",
        random_state=42
    ),
    param_distributions=xgb_params,
    n_iter=15,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_random.fit(X_train, y_train)

print("Best XGBoost Parameters:")
print(xgb_random.best_params_)

xgb_model = xgb_random.best_estimator_

xgb_predictions = xgb_model.predict(X_test)

xgb_results = evaluate_model(
    "XGBoost (Tuned)",
    y_test,
    xgb_predictions
)

# Comparison Between all models

In [ ]:
comparison = pd.DataFrame([
    baseline_results,
    lr_results,
    ridge_results,
    rf_results,
    xgb_results
])

comparison

In [ ]:
comparison = comparison.sort_values(by="RMSE").reset_index(drop=True)

# Ranking
comparison.index = comparison.index + 1
comparison.index.name = "Rank"

# Display comparison table with highlighted best metrics
display(
    comparison.style
    .format({
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "R2": "{:.3f}",
        "MAPE": "{:.2f}%"
    })
    .highlight_min(subset=["MAE", "RMSE", "MAPE"], color="#c8e6c9")
    .highlight_max(subset=["R2"], color="#c8e6c9")
)

# Display the selected model
best_model = comparison.iloc[0]

print("-" * 60)
print(" Final Selected Model")
print("-" * 60)
print(f"Model : {best_model['Model']}")
print(f"RMSE  : {best_model['RMSE']:.3f}")
print(f"R²    : {best_model['R2']:.3f}")
print(f"MAE   : {best_model['MAE']:.3f}")
print(f"MAPE  : {best_model['MAPE']:.2f}%")
print("-" * 60)

### Final Model Selection

Five forecasting models were evaluated using MAE, RMSE, MAPE, and R².

Although the tuned XGBoost model achieved the lowest MAE and MAPE, Ridge Regression achieved the lowest RMSE and the highest R² score.

Since demand forecasting is more sensitive to large prediction errors that may result in stock-outs or excess inventory, RMSE was selected as the primary evaluation metric. Based on this criterion, Ridge Regression was selected as the final forecasting model.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(
    y_test.values[:200],
    label="Actual Demand",
    linewidth=2
)

plt.plot(
    ridge_predictions[:200],
    label="Predicted Demand",
    linewidth=2
)

plt.title("Actual vs Predicted Demand (Ridge Regression)")
plt.xlabel("Test Samples")
plt.ylabel("Demand")

plt.legend()

plt.tight_layout()

plt.savefig("../outputs/figures/forecast_vs_actual.png")

plt.show()

In [ ]:
residuals = y_test - ridge_predictions
plt.hist(
    residuals,
    bins=30,
    edgecolor="black"
)

plt.title("Residual Distribution")

plt.xlabel("Residual")

plt.ylabel("Frequency")

plt.tight_layout()

plt.savefig("../outputs/figures/residual_distribution.png")

plt.show()

In [ ]:
import pickle

with open("../models/best_model.pkl", "wb") as file:
    pickle.dump(ridge_model, file)

print("Best model saved successfully!")

# **Model Training Summary**

The objective of this notebook was to develop and evaluate machine learning models capable of forecasting product demand eight weeks in advance for each supermarket–SKU combination. To ensure a realistic forecasting scenario, the dataset was divided using a chronological train-test split, preventing temporal data leakage and allowing the models to be evaluated on future observations.

A persistence baseline was established as a benchmark before training four machine learning models: Linear Regression, Ridge Regression, Random Forest, and XGBoost. Hyperparameter optimization for the tree-based models was performed using RandomizedSearchCV to identify parameter combinations that provided the best generalization performance.

Model performance was assessed using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), Mean Absolute Percentage Error (MAPE), and the Coefficient of Determination (R²). Since demand forecasting directly impacts inventory planning, RMSE was selected as the primary evaluation metric because it penalizes large prediction errors that can lead to stock-outs or excess inventory.

Among the evaluated approaches, Ridge Regression achieved the lowest RMSE and the highest R² score, while the tuned XGBoost model produced the lowest MAE and MAPE. As the difference between these models was marginal, Ridge Regression was selected as the final forecasting model due to its superior RMSE, simpler implementation, and strong generalization performance.

Overall, the developed forecasting pipeline demonstrates a complete end-to-end machine learning workflow, encompassing data preprocessing, feature engineering, model development, hyperparameter optimization, performance evaluation, and model selection. The resulting model provides a reliable foundation for supporting data-driven demand planning and inventory management decisions.

## Generate Forecast File

In [ ]:
import joblib

# Load trained model
best_model = joblib.load("../models/best_model.pkl")

# Generate forecasts
forecast_predictions = best_model.predict(X_test)

# Load encoders
sku_encoder = joblib.load("../models/sku_encoder.pkl")
supermarket_encoder = joblib.load("../models/supermarket_encoder.pkl")

# Create forecast dataframe
forecast_df = pd.DataFrame({
    "Date": test_df["date"].reset_index(drop=True),
    "SKU": test_df["sku"].reset_index(drop=True),
    "Supermarket": test_df["supermarket"].reset_index(drop=True),
    "Actual Demand": y_test.reset_index(drop=True),
    "Forecasted Demand": forecast_predictions.round(2)
})

# Decode categorical values
forecast_df["SKU"] = sku_encoder.inverse_transform(forecast_df["SKU"])
forecast_df["Supermarket"] = supermarket_encoder.inverse_transform(
    forecast_df["Supermarket"]
)

# Forecast Error
forecast_df["Forecast Error"] = (
    forecast_df["Actual Demand"] -
    forecast_df["Forecasted Demand"]
).round(2)

# Absolute Error
forecast_df["Absolute Error"] = (
    forecast_df["Forecast Error"]
    .abs()
)

# Display sample forecasts
forecast_df.head()

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

# Generate Demand Forecasts Using the Final Selected Model

# Load trained model and label encoders
best_model = joblib.load("../models/best_model.pkl")
sku_encoder = joblib.load("../models/sku_encoder.pkl")
supermarket_encoder = joblib.load("../models/supermarket_encoder.pkl")

# Generate demand forecasts
forecast_predictions = best_model.predict(X_test)

forecast_predictions = np.maximum(forecast_predictions, 0)

# Create forecast dataframe
forecast_df = pd.DataFrame({
    "Date": test_df["date"].values,
    "Supermarket": supermarket_encoder.inverse_transform(test_df["supermarket"]),
    "SKU": sku_encoder.inverse_transform(test_df["sku"]),
    "Actual Demand": y_test.values,
    "Forecasted Demand": np.round(forecast_predictions, 2)
})

# Calculate forecasting errors
forecast_df["Forecast Error"] = (
    forecast_df["Actual Demand"] -
    forecast_df["Forecasted Demand"]
).round(2)

forecast_df["Absolute Error"] = (
    forecast_df["Forecast Error"].abs()
)

# Sort for easier analysis
forecast_df.sort_values(
    by=["Date", "Supermarket", "SKU"],
    inplace=True
)

forecast_df.reset_index(drop=True, inplace=True)

# Save forecast file
os.makedirs("../outputs/predictions", exist_ok=True)

forecast_path = "../outputs/predictions/forecast.csv"

forecast_df.to_csv(
    forecast_path,
    index=False
)

# Display summary
print("-" * 60)
print("Forecast Generation Completed Successfully")
print("-" * 60)
print(f"Total Forecasts Generated : {len(forecast_df)}")
print(f"Forecast File Saved To    : {forecast_path}")
print("-" * 60)

forecast_df.head(10)